In [ ]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
  api_key=userdata.get("TOGETHER_API_KEY"),
  base_url="https://api.together.xyz/v1",
)

completion = client.chat.completions.create(
  model= "meta-llama/Meta-Llama-3.1-70B-Instruct-Turbo",
  messages=[
    #{"role": "user", "content": "Write me the code to count the number of A's in word Banana"},
    {"role": "user", "content": "Implement fizz buzz in Python"}
  ]
)

result = completion.choices[0].message.content

In [ ]:
import re

def extract_code_block(text):
    pattern = r'```(\w*)\n(.*?)```'
    match = re.search(pattern, text, re.DOTALL)

    if not match:
        return None, None

    # Extract language and code content
    language = match.group(1).lower() or None  # If no language specified, return None
    code_content = match.group(2).strip()

    return language, code_content

In [ ]:
print(extract_code_block(result)[0])

python


In [ ]:
print(extract_code_block(result)[1])

def fizz_buzz(n):
    """
    Prints the Fizz Buzz sequence up to n.

    Args:
        n (int): The upper limit of the sequence.
    """
    for i in range(1, n+1):
        if i % 3 == 0 and i % 5 == 0:
            print("FizzBuzz")
        elif i % 3 == 0:
            print("Fizz")
        elif i % 5 == 0:
            print("Buzz")
        else:
            print(i)

# Example usage:
fizz_buzz(20)


In [ ]:
!pip install e2b-code-interpreter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.7/81.7 kB 6.5 MB/s eta 0:00:00


In [ ]:
from e2b_code_interpreter import Sandbox

extracted_code = extract_code_block(result)[1]

sbx = Sandbox(api_key= userdata.get("E2B_API_KEY")) # By default the sandbox is alive for 5 minutes
execution = sbx.run_code(extracted_code) # Execute Python inside the sandbox
print(execution.logs)
print(execution.error)

sbx.kill()

Logs(stdout: ['1\n2\nFizz\n4\nBuzz\nFizz\n7\n8\nFizz\nBuzz\n11\nFizz\n13\n14\nFizzBuzz\n16\n17\nFizz\n19\nBuzz\n'], stderr: [])
None


In [ ]:
print(execution.logs.stdout[0])

1
2
Fizz
4
Buzz
Fizz
7
8
Fizz
Buzz
11
Fizz
13
14
FizzBuzz
16
17
Fizz
19
Buzz



In [ ]:
chat_history =[
    {"role": "system", "content": """
    Write Python code that you can run to answer the user's requests.
    Always trust the output of the code as true and summarize the output of the code to the user.
    """},
    {"role": "user", "content": "Get the stats for the GitHub user rajkstats."}

  ]

sbx = Sandbox(
    api_key = userdata.get("E2B_API_KEY")
)

while True:

  completion = client.chat.completions.create(
    model="meta-llama/Llama-3.3-70B-Instruct-Turbo-Free",
    messages=chat_history
  )
  response = completion.choices[0].message.content
  print(response)
  chat_history.append({"role": "system", "content": response})
  language, code = extract_code_block(response)
  if code:
    execution = sbx.run_code(code) # Execute Python inside the sandbox
    print(execution.logs.stdout)
    chat_history.append({
        "role": "assistant",
        "content": "Tell me about the GitHub user rajkstats. You can use the GitHub API. What do you think about his coding skills?"
    })
    if execution.error:
      print(execution.error)
      chat_history.append({
          "role": "assistant",
          "content": execution.error
      })
  else:
    break

```python
import requests

def get_github_user_stats(username):
    url = f"https://api.github.com/users/{username}"
    response = requests.get(url)
    if response.status_code == 200:
        user_data = response.json()
        return {
            "username": user_data["login"],
            "name": user_data["name"],
            "public_repos": user_data["public_repos"],
            "followers": user_data["followers"],
            "following": user_data["following"],
        }
    else:
        return None

username = "rajkstats"
user_stats = get_github_user_stats(username)

if user_stats:
    print(f"Username: {user_stats['username']}")
    print(f"Name: {user_stats['name']}")
    print(f"Public Repositories: {user_stats['public_repos']}")
    print(f"Followers: {user_stats['followers']}")
    print(f"Following: {user_stats['following']}")
else:
    print(f"Failed to retrieve stats for user {username}")
```

When you run this code, it will retrieve the stats for the GitHub user "ra

In [ ]:
chat_history

[{'role': 'system',
  'content': "Write Python code that you can run to answer the user's requests. Always trust the output of code as true."},
 {'role': 'user',
  'content': 'How many days are there until the first day of summer?'},
 {'role': 'system',
  'content': '```python\nfrom datetime import datetime, date\n\ndef days_until_summer():\n    # Define the first day of summer\n    summer_start = date(date.today().year, 6, 20)\n\n    # If the first day of summer has already passed this year, calculate for next year\n    if date.today() > summer_start:\n        summer_start = date(date.today().year + 1, 6, 20)\n\n    # Calculate the difference between today and the first day of summer\n    time_to_summer = summer_start - date.today()\n\n    return time_to_summer.days\n\nprint(f"There are {days_until_summer()} days until the first day of summer.")\n```'},
 {'role': 'assistant',
  'content': 'There are 151 days until the first day of summer.\n'},
 {'role': 'system',
  'content': 'This co